### 1.1.13. Operations on Sets of Tuples

Multi-dimensional sets — sets whose members are ordered pairs, triples, or longer tuples — support
the same binary operators (`union`, `inter`, `diff`, `symdiff`, `cross`) and the same membership
and cardinality functions (`in`, `within`, `card`) that are available on simple (1-D) sets.
Two additional constructs become especially important in the tuple context:

- **Sliced indexing** — fixing one or more components of a tuple to extract a lower-dimensional
  slice: `{(i,j) in LINKS}` with `i` already bound selects all `j` reachable from `i`.
- **The `setof` operator** — constructs a new set by projecting or transforming the tuples of
  an existing set: `setof {(i,j,p) in ROUTES} (i,j)` yields the 2-D set of origin–destination
  pairs that appear in at least one route.

This section covers both topics systematically, with emphasis on the dimension rules that govern
when each operator is applicable.

In [ ]:
# ── Grammar summary (BNF-style, informational) ───────────────────────────────
#
# Binary operators on tuple sets (dimensions must match for union/inter/diff/symdiff):
#
#   TUPLE_SET_A  union    TUPLE_SET_B   → same-dimension set, members in either
#   TUPLE_SET_A  inter    TUPLE_SET_B   → same-dimension set, members in both
#   TUPLE_SET_A  diff     TUPLE_SET_B   → same-dimension set, members in A not B
#   TUPLE_SET_A  symdiff  TUPLE_SET_B   → same-dimension set, members in exactly one
#   SET_A        cross    SET_B         → dim(A)+dim(B) set, all combined tuples
#
# Membership and subset tests:
#
#   (expr1, expr2)         in     TUPLE_SET   → Boolean
#   (expr1, expr2, expr3)  in     TUPLE_SET   → Boolean
#   TUPLE_SET_A  within    TUPLE_SET_B         → Boolean
#
# The setof operator:
#
#   setof { indexing-expression } member-expression
#
#   indexing-expression : one or more  dummy-var in SET  phrases,
#                         optionally with a filter condition after ':'
#   member-expression   : an expression or parenthesised tuple (e1, e2, ...)
#                         whose dimension determines the dimension of the result
#
# Iterated union / inter over an indexed collection of tuple sets:
#
#   union {p in PROD} LINKS[p]    → union of all 2-D sets LINKS[p]
#   inter {p in PROD} LINKS[p]    → intersection of all 2-D sets LINKS[p]

#### Dimension Rules

Every tuple set has a fixed **dimension** — the number of components in each member.
A set of pairs has dimension 2; a set of triples has dimension 3; and so on.
The AMPL dimension rules for set operators are:

| Operator | Dimension requirement | Result dimension |
|---|---|---|
| `union` | Both operands must have the same dimension $d$ | $d$ |
| `inter` | Both operands must have the same dimension $d$ | $d$ |
| `diff` | Both operands must have the same dimension $d$ | $d$ |
| `symdiff` | Both operands must have the same dimension $d$ | $d$ |
| `cross` | Operands may have any dimensions $d_1$, $d_2$ | $d_1 + d_2$ |
| `setof {…} (e1,…,ek)` | No constraint on the indexing set | $k$ (number of components in the member expression) |

Violating the dimension rule for `union`, `inter`, `diff`, or `symdiff` is a type error that
AMPL reports at model-generation time. For example, attempting
`LINKS union ROUTES` where `LINKS` is 2-D and `ROUTES` is 3-D is immediately rejected.

Compound sets (dimension ≥ 2) cannot be declared `ordered` or `circular`. Consequently,
the position-dependent functions `first`, `last`, `next`, `prev`, `ord`, and `ord0` are
never valid on tuple sets, regardless of how they were constructed
(Fourer, Gay, Kernighan, §6.4).

#### Conceptual Explanation

**Set operators on tuple sets** work identically to their 1-D counterparts — membership of a tuple
$(i_1, \ldots, i_d)$ is determined by whether that exact tuple appears in the operand.
Given two 2-D sets $A$ and $B$:

$$A \cup B = \{(i,j) : (i,j) \in A \text{ or } (i,j) \in B\}$$
$$A \cap B = \{(i,j) : (i,j) \in A \text{ and } (i,j) \in B\}$$
$$A \setminus B = \{(i,j) : (i,j) \in A \text{ and } (i,j) \notin B\}$$

**`cross` on multi-dimensional operands** extends dimension. If $A$ is $d_1$-dimensional
and $B$ is $d_2$-dimensional:
$$A \times B = \{(a_1, \ldots, a_{d_1}, b_1, \ldots, b_{d_2}) : (a_1,\ldots,a_{d_1}) \in A,\; (b_1,\ldots,b_{d_2}) \in B\}$$

So `PAIRS cross PROD` where `PAIRS` is 2-D and `PROD` is 1-D yields a 3-D set.

**`setof`** is a set-construction operator analogous to the iterated `sum` for arithmetic.
It evaluates a member expression for every element of an indexing set and collects distinct
results. The dimension of the result equals the number of components in the member expression:

- `setof {(i,j,p) in ROUTES} p` — 1-D: projects the product component
- `setof {(i,j,p) in ROUTES} (i,j)` — 2-D: projects the origin–destination pair
- `setof {i in 1..5} (i, i^2)` — 2-D: constructs pairs $(i, i^2)$

Duplicate results are automatically suppressed; the resulting set contains each distinct
member at most once, regardless of how many indexing tuples map to it.

#### The `setof` Operator — Reference

| Form | Result dimension | Description |
|---|---|---|
| `setof {i in S} expr` | 1 | Set of scalar values of `expr` over `S` |
| `setof {i in S} (e1, e2)` | 2 | Set of pairs constructed from `S` |
| `setof {(i,j) in LINKS} i` | 1 | First-component projection (set of origins) |
| `setof {(i,j) in LINKS} j` | 1 | Second-component projection (set of destinations) |
| `setof {(i,j) in LINKS} (i,j)` | 2 | Identity projection (copy of `LINKS`) |
| `setof {(i,j,p) in ROUTES} p` | 1 | Third-component projection |
| `setof {(i,j,p) in ROUTES} (i,j)` | 2 | First-two-component projection |
| `setof {(i,j,p) in ROUTES: p = 'bands'} (i,j)` | 2 | Filtered projection: only 'bands' routes |
| `setof {i in S, j in T: cond} (i,j)` | 2 | Filtered Cartesian construction |
| `setof {k in CATEG} (setof {i in PEOPLE} type[i,k])` | varies | Indexed collection of sets |

**Formal grammar** (from AMPL Reference Manual, §A.3):

```
setof  indexing  member
```

where `member` is either a single expression or a parenthesised, comma-separated list of
expressions. The dimension of the result equals the number of components in `member`.
Duplicate members are ignored. The `setof` operator has precedence level 11 in the AMPL
operator table — tighter than all set-algebra operators — so no parentheses are needed around
its arguments when it appears as an operand to `union`, `inter`, etc.

#### Example 1 — Union, Inter, Diff on 2-D Arc Sets

Two sparse arc sets over the same `ORIG cross DEST` domain are combined with the standard
binary operators. Both operands are 2-D, so all operators are dimension-compatible.

In [ ]:
# Two sparse arc sets — both are subsets of ORIG cross DEST
set ORIG;
set DEST;

set LINKS_A within ORIG cross DEST;   # arcs in network A
set LINKS_B within ORIG cross DEST;   # arcs in network B

# Union: arcs that appear in network A or B (or both)
set ALL_LINKS   = LINKS_A union  LINKS_B;  # 2-D result

# Intersection: arcs present in BOTH networks
set SHARED      = LINKS_A inter  LINKS_B;  # 2-D result

# Difference: arcs only in A
set ONLY_A      = LINKS_A diff   LINKS_B;  # 2-D result

# Symmetric difference: arcs in exactly one network
set EXCLUSIVE   = LINKS_A symdiff LINKS_B; # 2-D result

# Membership test for a specific pair
# Evaluates to 1 (true) if the pair (i,j) is in ALL_LINKS
param i symbolic;
param j symbolic;
param active = if (i,j) in ALL_LINKS then 1 else 0;

#### Example 2 — `cross` on Multi-Dimensional Sets

When at least one operand of `cross` is itself multi-dimensional, the result dimension is the
sum of the two operand dimensions. This example builds a 3-D set from a 2-D set and a 1-D
set, then from two 2-D sets to get a 4-D set.

In [ ]:
set ORIG;
set DEST;
set PROD;

# 2-D set of (origin, destination) pairs
set LINKS within ORIG cross DEST;      # dimension 2

# cross with a 1-D set: result is 3-D (origin, destination, product)
set ROUTES = LINKS cross PROD;         # dimension 2 + 1 = 3

# Equivalent declaration using within and a cross expression
set ROUTES2 within ORIG cross DEST cross PROD;  # same structure, but sparse

# cross of two 2-D sets: result is 4-D
set LINKS_PAIRS = LINKS cross LINKS;   # dimension 2 + 2 = 4
# each member is a quadruple (i1, j1, i2, j2)

# Parameters and variables indexed over the 3-D set
param cost  {ROUTES} >= 0;
var   Trans {ROUTES} >= 0;

minimize Total_Cost:
    sum {(i,j,p) in ROUTES} cost[i,j,p] * Trans[i,j,p];

# Sliced sum: for a fixed origin i and product p, sum over all destinations j
subject to Supply {i in ORIG, p in PROD}:
    sum {(i,j,p) in ROUTES} Trans[i,j,p] = 0;  # placeholder RHS

**Note on sliced indexing.** In the `Supply` constraint above, `i` and `p` are bound by the outer
indexing `{i in ORIG, p in PROD}`. Inside the sum, `{(i,j,p) in ROUTES}` uses those bound
values to extract only those `j` such that `(i,j,p)` is a triple in `ROUTES`. AMPL applies this
slice efficiently without generating the full Cartesian product `DEST cross PROD` first
(Fourer, Gay, Kernighan, §6.3). This is the canonical pattern for sums over sparse
multi-dimensional sets.

#### Example 3 — The `setof` Operator: Projection and Filtering

This example shows the three canonical uses of `setof` on a 3-D route set:
full projection to recover component sets, partial projection (2-D slice), and
filtered projection.

In [ ]:
# Multicommodity network: routes are (origin, destination, product) triples
set ROUTES dimen 3;   # members supplied from data; domain not declared in model

# ── Full projection: recover component sets from ROUTES ────────────────────

# Derive PROD as the set of products that appear in at least one route
set PROD  = setof {(i,j,p) in ROUTES} p;         # 1-D: {"bands","coils",...}

# Derive ORIG as the set of origins that appear in at least one route
set ORIG  = setof {(i,j,p) in ROUTES} i;         # 1-D

# Derive DEST as the set of destinations that appear in at least one route
set DEST  = setof {(i,j,p) in ROUTES} j;         # 1-D

# ── Partial projection: 2-D link set from 3-D routes ──────────────────────

# LINKS = all (origin, destination) pairs for which at least one product is shipped
set LINKS = setof {(i,j,p) in ROUTES} (i,j);    # 2-D

# ── Filtered projection ────────────────────────────────────────────────────

# Links that carry product 'bands' only
set BANDS_LINKS = setof {(i,j,p) in ROUTES: p = 'bands'} (i,j);  # 2-D

# Origins that ship product 'coils' to destination 'DET'
set COIL_ORIG = setof {(i,j,p) in ROUTES: p = 'coils' and j = 'DET'} i;  # 1-D

**Why use `setof` to derive component sets?**
When a model receives a sparse 3-D set `ROUTES` as its primary data, requiring the user to
also supply `ORIG`, `DEST`, and `PROD` separately introduces a redundancy and a potential
inconsistency. Using `setof` to derive these sets guarantees they contain exactly the elements
that actually appear in the data, with no possibility of mismatch (Fourer, Gay, Kernighan, §6.4).

#### Example 4 — `setof` to Derive Index Sets for Constraints

A practical pattern: use `setof` to construct the exact index domain of a constraint from
the data, rather than indexing over a Cartesian product and filtering with a membership
condition.

In [ ]:
# Multicommodity transportation model — full model using setof-derived index sets

set ROUTES dimen 3;                               # (orig, dest, prod) triples from data

set ORIG   = setof {(i,j,p) in ROUTES} i;
set DEST   = setof {(i,j,p) in ROUTES} j;
set PROD   = setof {(i,j,p) in ROUTES} p;
set LINKS  = setof {(i,j,p) in ROUTES} (i,j);

# Pairs (dest, prod) for which there is positive demand
param demand {DEST, PROD} >= 0 default 0;
set DEMAND = setof {j in DEST, p in PROD: demand[j,p] > 0} (j,p);  # 2-D sparse

param supply {ORIG, PROD} >= 0 default 0;
param cost   {ROUTES}     >= 0;

var Trans {ROUTES} >= 0;

minimize Total_Cost:
    sum {(i,j,p) in ROUTES} cost[i,j,p] * Trans[i,j,p];

# Supply constraint: indexed over (orig, prod) pairs; sliced sum over j
subject to Supply {i in ORIG, p in PROD}:
    sum {(i,j,p) in ROUTES} Trans[i,j,p] = supply[i,p];

# Demand constraint: indexed over the sparse DEMAND set
# only generates constraints for (j,p) with positive demand
subject to Demand {(j,p) in DEMAND}:
    sum {(i,j,p) in ROUTES} Trans[i,j,p] = demand[j,p];

#### Example 5 — Iterated `union` and `inter` over Tuple Sets

When a model declares an indexed collection of tuple sets — for example, `set LINKS {PROD}` —
the iterated `union` and `inter` operators reduce the collection to a single set, just as
iterated `sum` reduces a numeric array to a scalar.

This pattern arises in network models where each product has its own set of valid arcs.

In [ ]:
set PROD;
set ORIG;
set DEST;

# Each product has its own sparse set of valid arcs
set LINKS {PROD} within ORIG cross DEST;

# ── Iterated union ─────────────────────────────────────────────────────────
# All arcs used by at least one product
set ALL_LINKS = union {p in PROD} LINKS[p];      # 2-D; empty when PROD is empty

# ── Iterated inter ─────────────────────────────────────────────────────────
# Arcs used by EVERY product (requires PROD to be non-empty)
set SHARED_LINKS = inter {p in PROD} LINKS[p];   # 2-D; error when PROD is empty

# ── setof version of union ─────────────────────────────────────────────────
# Equivalent construction of ALL_LINKS using setof:
set ALL_LINKS2 = setof {p in PROD, (i,j) in LINKS[p]} (i,j);  # 2-D; same result

# ── Using ALL_LINKS in a variable and objective ────────────────────────────
param trans_cost {(i,j) in ALL_LINKS} >= 0;
var   Flow       {(i,j) in ALL_LINKS} >= 0;

minimize Arc_Cost:
    sum {(i,j) in ALL_LINKS} trans_cost[i,j] * Flow[i,j];

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
Iterated <code>inter</code> over an empty index set is undefined and causes an error at
model-generation time: <em>"inter over empty set is not defined"</em>. This applies to
iterated intersection over any tuple set, just as for simple sets. Always ensure the
index set <code>PROD</code> (or whatever drives the iteration) is non-empty before using
<code>inter {p in PROD} LINKS[p]</code>. Iterated <code>union</code> over an empty index
set, by contrast, safely returns the empty set.</div>

#### Example 6 — `setof` with a Filtered Condition

A condition placed after `:` inside the `setof` indexing expression restricts which tuples
contribute to the result. This is used to derive sparse index sets from existing sets and
parameters.

In [ ]:
set DEST;
set PROD;

param demand {DEST, PROD} >= 0;

# ── setof with condition on a parameter ───────────────────────────────────

# (j,p) pairs where demand is strictly positive
set DEMAND = setof {j in DEST, p in PROD: demand[j,p] > 0} (j,p);

# Destinations that receive at least one unit of product 'bands'
set BANDS_DEST = setof {j in DEST: demand[j,'bands'] > 0} j;

# ── setof with condition on set membership ─────────────────────────────────

set ORIG;
set ROUTES dimen 3;    # (orig, dest, prod) triples

# All (orig, dest) pairs for routes whose product is 'bands'
set BANDS_LINKS = setof {(i,j,p) in ROUTES: p = 'bands'} (i,j);

# All products routed through a specific origin-destination pair ('GARY','DET')
set GARY_DET_PROD = setof {(i,j,p) in ROUTES: i = 'GARY' and j = 'DET'} p;

# ── setof to build a 2-D set of same-origin transfer pairs ────────────────

# All pairs of distinct origins (for modelling transfers between origins)
set TRANSF = setof {i1 in ORIG, i2 in ORIG: i1 <> i2} (i1,i2);
# Equivalent: set TRANSF = {i1 in ORIG, i2 in ORIG: i1 <> i2};

#### Membership Test for Tuples

The `in` operator tests whether a specific tuple is a member of a set. The tuple is written
as a parenthesised, comma-separated list of expressions.

In [ ]:
set LINKS  within {ORIG, DEST};
set ROUTES within {ORIG, DEST, PROD};

# Boolean membership test for a pair
param link_exists {i in ORIG, j in DEST} = if (i,j) in LINKS then 1 else 0;

# Boolean membership test for a triple
param route_exists {i in ORIG, j in DEST, p in PROD} =
    if (i,j,p) in ROUTES then 1 else 0;

# Use in a constraint: ship only on declared routes
# (the idiomatic AMPL form -- index the variable over ROUTES directly)
var Trans {ROUTES} >= 0;

# Alternatively, use membership as a filter inside a sum:
param cost {ROUTES} >= 0;
subject to Supply_All {i in ORIG}:
    sum {(i,j,p) in ROUTES} Trans[i,j,p] >= 0;  # sliced; preferred over membership test

# Non-membership test:
param no_direct_link {i in ORIG, j in DEST} = if (i,j) not in LINKS then 1 else 0;

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
When constructing a sum over a sparse multi-dimensional set, use the sliced indexing
form <code>sum {(i,j,p) in ROUTES}</code> rather than the filtered form
<code>sum {j in DEST, p in PROD: (i,j,p) in ROUTES}</code>. The sliced form lets AMPL
use its internal index structure to locate only the relevant tuples directly, without
generating the full Cartesian product of <code>DEST</code> and <code>PROD</code> and
testing each element for membership. For large sets this difference in efficiency can be
decisive (Fourer, Gay, Kernighan, §6.3).</div>

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
When a model receives a sparse high-dimensional set as its primary data — for example,
a set of <code>(origin, destination, product)</code> triples — use <code>setof</code>
to derive the component 1-D sets (<code>ORIG</code>, <code>DEST</code>, <code>PROD</code>)
and any needed 2-D projections (<code>LINKS</code>). This ensures consistency: the derived
sets contain exactly the elements that appear in the data, and the modeler does not have
to maintain them separately. See Fourer, Gay, Kernighan §6.4 for the canonical example.</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
The dimension of a <code>setof</code> result is determined by the member expression, not
by the indexing set. Writing <code>setof {(i,j,p) in ROUTES} p</code> produces a 1-D set
(scalars), while <code>setof {(i,j,p) in ROUTES} (i,p)</code> produces a 2-D set (pairs).
If you then pass the result to an operator that expects a different dimension — for example,
taking the union of a 1-D projection with a 2-D set — AMPL will report a type error at
model-generation time. Always verify that the member expression has the intended number
of components.</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
Attempting to form <code>union</code>, <code>inter</code>, <code>diff</code>, or
<code>symdiff</code> of two sets with different dimensions is a type error. A common
mistake is writing <code>LINKS union ROUTES</code> when <code>LINKS</code> is 2-D and
<code>ROUTES</code> is 3-D. Use <code>cross</code> to combine sets of differing dimensions,
or <code>setof</code> to project to a common dimension first.</div>

#### amplpy Equivalents

In amplpy, tuple set operations are performed inside AMPL via `ampl.eval()`. The results are
read back into Python as lists of tuples via `members()` or as amplpy DataFrames via
`get_values()`. Python's native set operations (`|`, `&`, `-`, `^`) on the Python-side
representation can replicate the same logic when the computation is needed only in Python.

| AMPL operation | amplpy approach |
|---|---|
| `set LINKS = LINKS_A union LINKS_B;` | Define in model, or `ampl.eval("set ALL_LINKS = LINKS_A union LINKS_B;")` |
| Retrieve 2-D set members as list of tuples | `ampl.set['LINKS'].members()` — returns `[('GARY','DET'), ...]` |
| Retrieve 2-D set as pandas DataFrame | `ampl.set['LINKS'].get_values().to_pandas()` |
| Assign a 2-D set from a list of tuples | `ampl.set['LINKS'].set_values([('GARY','DET'), ('CLEV','FRA')])` |
| Assign a 3-D set from a list of triples | `ampl.set['ROUTES'].set_values([('GARY','DET','bands'), ...])` |
| Cardinality of a tuple set | `ampl.set['LINKS'].size()` or `ampl.get_value('card(LINKS)')` |
| Boolean membership test in Python | `('GARY','DET') in set(ampl.set['LINKS'].members())` |
| Evaluate `setof` expression | `ampl.eval("display setof {(i,j,p) in ROUTES} (i,j);")` |
| Python-side union of two 2-D sets | `set(ampl.set['LINKS_A'].members()) | set(ampl.set['LINKS_B'].members())` |

In [ ]:
# amplpy: working with multi-dimensional sets

# Assume ampl object is already initialised with the model above loaded.

# ── Assigning a 2-D set from Python ───────────────────────────────────────
ampl.eval("set LINKS_A dimen 2; set LINKS_B dimen 2;")
ampl.set["LINKS_A"].set_values([("GARY", "DET"), ("GARY", "FRA"), ("CLEV", "DET")])
ampl.set["LINKS_B"].set_values([("GARY", "FRA"), ("PITT", "DET"), ("PITT", "FRA")])

# ── Reading back a derived set ─────────────────────────────────────────────
ampl.eval("set ALL_LINKS = LINKS_A union LINKS_B;")
all_links = ampl.set["ALL_LINKS"].members()
# all_links is a list of tuples:
# [("GARY","DET"), ("GARY","FRA"), ("CLEV","DET"), ("PITT","DET"), ("PITT","FRA")]

# ── Python-side set operations for Python-only logic ──────────────────────
py_a = set(ampl.set["LINKS_A"].members())
py_b = set(ampl.set["LINKS_B"].members())

py_union = py_a | py_b          # same as LINKS_A union LINKS_B
py_inter = py_a & py_b          # same as LINKS_A inter LINKS_B
py_diff  = py_a - py_b          # same as LINKS_A diff LINKS_B

# ── Assigning a 3-D set from Python ───────────────────────────────────────
ampl.eval("set ROUTES dimen 3;")
ampl.set["ROUTES"].set_values([
    ("GARY", "DET", "bands"),
    ("GARY", "FRA", "coils"),
    ("CLEV", "DET", "bands"),
    ("PITT", "FRA", "plate"),
])

# ── Evaluating setof inside AMPL and reading the result ───────────────────
ampl.eval("""
    set LINKS_FROM_ROUTES = setof {(i,j,p) in ROUTES} (i,j);
""")
links = ampl.set["LINKS_FROM_ROUTES"].members()
# [("GARY","DET"), ("GARY","FRA"), ("CLEV","DET"), ("PITT","FRA")]

#### Edge Cases

**Edge case 1: Union and intersection of a tuple set with itself and with the empty set.**

The standard set-algebra identities hold for tuple sets:
- $A \cup \emptyset = A$
- $A \cap \emptyset = \emptyset$
- $A \setminus A = \emptyset$
- $A \cup A = A$

No error is raised when one operand is an empty tuple set, provided both have the same dimension.
A model that uses `LINKS_A union LINKS_B` is correct even when `LINKS_B` is empty for a
particular data instance.

In [ ]:
set ORIG;
set DEST;

set LINKS_A within ORIG cross DEST;
set LINKS_B within ORIG cross DEST;  # may be empty for a given data instance

# Safe: union with an empty 2-D set yields the non-empty operand
set ALL_LINKS = LINKS_A union LINKS_B;    # = LINKS_A if LINKS_B is empty

# Safe: difference of a set with itself yields the empty set
set NONE = LINKS_A diff LINKS_A;          # always {}

# Safe: cardinality of an empty tuple set is 0
param n_shared = card(LINKS_A inter LINKS_B);  # = 0 if they are disjoint

# A constraint indexed over an empty tuple set generates zero instances — no error:
param flow_cost {LINKS_A inter LINKS_B} >= 0;
var   Shared_Flow {LINKS_A inter LINKS_B} >= 0;
subject to Shared_Cap {(i,j) in LINKS_A inter LINKS_B}:  # zero instances if intersection empty
    Shared_Flow[i,j] <= flow_cost[i,j];

**Edge case 2: `setof` over an empty indexing set returns the empty set.**

When the indexing set in a `setof` expression is empty, the result is an empty set of the
declared dimension. No error is raised. This means that derived sets built with `setof` are
safe even when the underlying data set is empty.

In [ ]:
# ROUTES is declared but may receive zero triples from data
set ROUTES dimen 3;

# All three derived sets are empty when ROUTES is empty:
set PROD  = setof {(i,j,p) in ROUTES} p;      # {} if ROUTES empty
set ORIG  = setof {(i,j,p) in ROUTES} i;      # {} if ROUTES empty
set LINKS = setof {(i,j,p) in ROUTES} (i,j);  # {} if ROUTES empty

# Model components indexed over empty derived sets generate zero instances — not errors:
param cost  {ROUTES}        >= 0;
var   Trans {ROUTES}        >= 0;
param supply {ORIG, PROD}   >= 0 default 0;

minimize Total_Cost:                           # objective sum = 0 when ROUTES empty
    sum {(i,j,p) in ROUTES} cost[i,j,p] * Trans[i,j,p];

subject to Supply {i in ORIG, p in PROD}:      # zero constraints when ORIG or PROD empty
    sum {(i,j,p) in ROUTES} Trans[i,j,p] = supply[i,p];

**Edge case 3: `cross` applied to a set of pairs — dimension accumulates.**

`cross` does not restrict its operands to being 1-D. When one or both operands are already
multi-dimensional, the result dimension is the sum. This behaviour allows building arbitrarily
high-dimensional sets, and is intentional. However, the resulting high-dimensional set can be
very large — `LINKS cross LINKS` has $|\text{LINKS}|^2$ members — so care is needed.

In [ ]:
set ORIG;  set DEST;  set PROD;

set LINKS   within ORIG cross DEST;         # 2-D
set ROUTES  = LINKS cross PROD;             # 3-D: (orig, dest, prod)

# Further cross: 4-D set of pairs-of-links
# Each member is a quadruple (i1,j1,i2,j2)
set LINK_PAIRS = LINKS cross LINKS;         # 4-D
# card(LINK_PAIRS) = card(LINKS)^2 — potentially large

# 5-D: (orig, dest, prod, orig, dest)
set FIVE_DIM = ROUTES cross LINKS;          # 3 + 2 = 5

# Preferred alternative when the full Cartesian product is not needed:
# use setof with a filter to build only the subset you actually need.
set SAME_ORIG_PAIRS = setof {(i1,j1) in LINKS, (i2,j2) in LINKS:
                              i1 = i2 and j1 <> j2} (i1,j1,i2,j2);  # 4-D, filtered

**Edge case 4: `setof` in a transitive-closure pattern.**

The AMPL Reference Manual (§A.6.2) documents a use of `setof` combined with iterated
`union` to compute a transitive closure — the set of all node pairs reachable by any number
of hops in a directed graph. This is an advanced use that demonstrates `setof` inside an
indexed set definition:

In [ ]:
# Transitive closure of a directed graph (AMPL Reference Manual, A.6.2)
set nodes;
set arcs within nodes cross nodes;   # direct arcs: 2-D
param max_iter integer > 0;

# step[s] = all pairs reachable in at most s hops
# step[1] = the original arc set
# step[s] = step[s-1] union all pairs (i,j) reachable via one intermediate node k
set step {s in 1..max_iter} dimen 2 =
    if s == 1
    then arcs
    else step[s-1] union setof {k in nodes,
                                 (i,k) in step[s-1],
                                 (k,j) in step[s-1]} (i,j);

# The transitive closure (all pairs reachable in up to max_iter hops):
set reach = step[max_iter];

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
Compound sets cannot be declared <code>ordered</code> or <code>circular</code>. This is
a hard restriction in AMPL: any set of dimension two or higher is always unordered.
Consequently, the position-dependent functions <code>first</code>, <code>last</code>,
<code>next</code>, <code>prev</code>, <code>ord</code>, and <code>ord0</code> are never
valid on tuple sets. If a model needs to iterate over links in a specific sequence,
the sequence must be encoded explicitly — for example, by introducing an integer index
parameter <code>seq {LINKS}</code> and sorting in Python before calling
<code>set_values</code>.</div>

#### Cross-References

- See also: [1.1.1. Set Declaration vs. Definition](<1_1_1 Set Declaration vs. Definition.ipynb>) — `=`, `default`, `:=` in set declarations
- See also: [1.1.4. Set Operations](<1_1_4 set_operations.ipynb>) — `union`, `inter`, `diff`, `symdiff`, `cross` on simple sets; iterated operators
- See also: [1.1.5. Unordered Sets](<1_1_5 unordered_sets.ipynb>) — why compound sets are always unordered; ordering restrictions
- Fourer, Gay, Kernighan, *AMPL: A Modeling Language for Mathematical Programming*, §6.4 (Operations on Sets of Tuples) and §6.3 (Sets of Longer Tuples)
- AMPL Reference Manual, §A.3 (set expression grammar), §A.6.2 (set operators and precedence)